# Volume 8, Chapter 8: Haar Measure and Weyl Integration (Python companion)

Random unitary matrices done right, Weyl's eigenvalue law tested, Haar random quantum states and cross-entropy benchmarking, and the zeros of the zeta function set against the eigenvalues of random matrices.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★) Sampling the Haar measure, right and wrong.** (a) Implement Theorem 8.7: a Haar random $U\in U(n)$ is the $Q$ of the $QR$ decomposition of a complex Gaussian matrix, *after* multiplying the columns of $Q$ by the phases of the diagonal of $R$. (b) Compare with the raw output of `numpy.linalg.qr`. Test both against two predictions of the chapter: the eigenvalue angles are uniform on the circle (Theorem 8.15 with $k=1$), and $\mathbb E|\operatorname{tr}U|^2=1$ (Exercise 8.4). (c) Check Weyl's formula for $n=2$: the distance between the two eigenangles has density $\frac{1-\cos x}{2\pi}$ on $(-\pi,\pi]$ (Exercise 8.6).

**P8.2 (★★) CS application: random quantum states and the Porter–Thomas law.** A quantum computer with $q$ qubits has states in $\C^N$, $N=2^q$. A *Haar random state* is the first column $\psi=Ue_1$ of a Haar random unitary, and random quantum circuits are designed to produce states that look Haar random. (a) Sample Haar random states without building $U$ (Exercise 8.2: normalize a Gaussian vector), and check that the measurement probabilities $p_x=|\psi_x|^2$, rescaled to $Np_x$, follow the exponential law $e^{-s}$ (the *Porter–Thomas* law), with $\mathbb E[N^2p_x^2]\to2$. (b) *Cross-entropy benchmarking*: if a device samples bit strings $x$ from the ideal distribution $p$, then $\mathbb E[Np_x]-1$ is close to $1$; if it outputs uniform noise, it is close to $0$. Simulate both and a mixture, and estimate the fidelity from samples. This is the statistic behind the 2019 quantum supremacy experiment.

**P8.3 (★★) Zeros of zeta against random matrices.** (a) Compute the zeros $\frac12+i\gamma_n$ for $100<n\le600$ with `mpmath.zetazero`, rescale them by $\tilde\gamma=\frac\gamma{2\pi}\log\frac\gamma{2\pi e}$ (mean spacing $1$, by the Riemann–von Mangoldt formula), and compare the histogram of consecutive spacings with that of the eigenvalues of Haar random matrices of $U(50)$ (rescaled by $\frac n{2\pi}$) and with the exponential law of independent points. (b) Compare the pair correlation of the zeros with $1-(\frac{\sin\pi u}{\pi u})^2$ (Corollary 8.16). (c) Check Heine's identity on the gap probability of Exercise 8.5: compute the Toeplitz determinant for $n=10$ and an arc of half-angle $\alpha$, and compare with a simulation.

## Solutions

### P8.1

In [ ]:
import numpy as np

rng = np.random.default_rng(8)

def haar_unitary(n, fix=True):
    Z = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(Z)
    if fix:
        d = np.diag(R)
        Q = Q * (d / abs(d))          # multiply column j by the phase of r_jj
    return Q

n, trials = 3, 40000
for fix in (True, False):
    angles, tr2 = [], []
    for _ in range(trials):
        U = haar_unitary(n, fix)
        angles.extend(np.angle(np.linalg.eigvals(U)))
        tr2.append(abs(np.trace(U)) ** 2)
    h, _ = np.histogram(angles, bins=8, range=(-np.pi, np.pi))
    print("phase fix" if fix else "raw numpy", "  E|tr U|^2 =", round(np.mean(tr2), 3))
    print("   eigenangle histogram (8 bins, uniform would be equal):", np.round(h / h.mean(), 2))

With the phase fix, $\mathbb E|\operatorname{tr}U|^2$ is $1$ up to sampling error and the eigenangles are uniform. Without it, the eigenangles avoid the neighbourhood of angle $0$ almost completely (the two middle bins are nearly empty) and $\mathbb E|\operatorname{tr}U|^2$ is visibly off, because the raw Householder $Q$ has a biased distribution. Nothing in the output of `numpy.linalg.qr` warns you; this is the bug Mezzadri's paper was written to prevent. (Recent versions of `scipy.stats.unitary_group` do the right thing.)

In [ ]:
# Weyl for n = 2: the distance x between the two eigenangles has density (1 - cos x) / (2 pi)
xs = []
for _ in range(60000):
    a, b = np.angle(np.linalg.eigvals(haar_unitary(2)))
    xs.append(np.angle(np.exp(1j * (a - b))))       # representative in (-pi, pi]
h, edges = np.histogram(xs, bins=12, range=(-np.pi, np.pi), density=True)
mid = (edges[1:] + edges[:-1]) / 2
print("   x      empirical   (1 - cos x)/(2 pi)")
for m, v in zip(mid, h):
    print(f"{m:6.2f}    {v:.4f}      {(1 - np.cos(m)) / (2 * np.pi):.4f}")
eps = 0.3
print("P(|x| < 0.3):", np.mean(np.abs(xs) < eps), " theory:", (eps - np.sin(eps)) / np.pi, " independent points:", eps / np.pi)

The histogram follows $\frac{1-\cos x}{2\pi}$, empty near $0$ and highest at $\pm\pi$: the two eigenvalues prefer to sit opposite each other. The probability of being within $0.3$ radians is about $0.0014$, against about $0.095$ for two independent points.

### P8.2

In [ ]:
def haar_state(N):
    z = (rng.standard_normal(N) + 1j * rng.standard_normal(N)) / np.sqrt(2)
    return z / np.linalg.norm(z)

for q in (2, 4, 8, 12):
    N = 2 ** q
    s = np.concatenate([N * abs(haar_state(N)) ** 2 for _ in range(max(1, 200000 // N))])
    print(f"q = {q:2d} qubits:  mean of N p = {s.mean():.3f},  mean of (N p)^2 = {np.mean(s**2):.3f},"
          f"  P(N p > 2) = {np.mean(s > 2):.4f}  (Porter-Thomas: 2, e^-2 = {np.exp(-2):.4f})")

The mean of $Np_x$ is exactly $1$ (the probabilities sum to $1$). Its second moment is $\frac{2N}{N+1}$, by the Beta law of Exercise 8.2, which tends to $2$, the second moment of the exponential law; so does the tail $\mathbb P(Np>2)\to e^{-2}$. Already at $12$ qubits the agreement is within sampling error. The probabilities of a random state are therefore very uneven: some bit strings are several times likelier than average, and that unevenness is what a test can detect.

In [ ]:
q = 12; N = 2 ** q
psi = haar_state(N); p = abs(psi) ** 2          # the ideal output distribution, known to the verifier
def xeb(samples):                                # linear cross-entropy benchmark
    return N * p[samples].mean() - 1
ideal = rng.choice(N, size=20000, p=p)
noise = rng.integers(0, N, size=20000)
for F in (1.0, 0.5, 0.1, 0.0):                  # a device that is ideal with probability F, noise otherwise
    mix = np.where(rng.random(20000) < F, ideal, noise)
    print(f"true fidelity {F:.1f}:  XEB estimate {xeb(mix):.3f}")

For ideal samples $\mathbb E[Np_x]=N\sum_xp_x^2\approx2$, so the estimate is close to $1$; for uniform noise it is $N\sum_x\frac1Np_x=1$, giving $0$; and a mixture gives the mixing weight, so the statistic measures how much of the device's output is signal. Computing $p_x$ requires simulating the circuit classically, which is feasible at $12$ qubits and was the hard part at $53$. The Porter–Thomas law, a consequence of the Haar measure, is what makes the test work.

### P8.3

In [ ]:
import mpmath, time
t0 = time.time()
gam = np.array([float(mpmath.zetazero(k).imag) for k in range(101, 601)])
print(f"{len(gam)} zeros from {gam[0]:.3f} to {gam[-1]:.3f}, computed in {time.time() - t0:.0f} s")
unf = gam / (2 * np.pi) * np.log(gam / (2 * np.pi * np.e))     # unfolded: mean spacing about 1
sz = np.diff(unf); sz = sz / sz.mean()

nU, reps = 50, 400
su = []
for _ in range(reps):
    th = np.sort(np.angle(np.linalg.eigvals(haar_unitary(nU))))
    su.extend(np.diff(np.r_[th, th[0] + 2 * np.pi]) * nU / (2 * np.pi))
su = np.array(su)

bins = np.linspace(0, 3, 13)
hz, _ = np.histogram(sz, bins, density=True); hu, _ = np.histogram(su, bins, density=True)
mid = (bins[1:] + bins[:-1]) / 2
print("spacing   zeta zeros   CUE(50)   independent e^-s")
for m, a, b in zip(mid, hz, hu):
    print(f"{m:5.2f}      {a:.3f}       {b:.3f}      {np.exp(-m):.3f}")
print("fraction of spacings below 0.25:  zeta", np.mean(sz < 0.25), " CUE", round(np.mean(su < 0.25), 4), " independent", round(1 - np.exp(-0.25), 4))

Both the zeros and the eigenvalues have very few small spacings, where independent points would have their highest density, and both histograms peak near $s\approx1$; the zeta column follows the CUE column within the noise of $500$ spacings. These are low zeros, around height $250$ to $900$; Odlyzko's computations near the $10^{12}$-th zero and beyond show the agreement getting better still, with second-order corrections that are themselves predicted by random matrix theory (Bogomolny and Keating).

In [ ]:
def pair_corr(x, umax=3.0, nb=15):
    d = []
    for i in range(len(x)):
        j = i + 1
        while j < len(x) and x[j] - x[i] < umax:
            d.append(x[j] - x[i]); j += 1
    h, e = np.histogram(d, np.linspace(0, umax, nb + 1))
    return (e[1:] + e[:-1]) / 2, h / (len(x) * (e[1] - e[0]))
uz, rz = pair_corr(unf)
print("  u    zeta pairs   1-(sin pi u/pi u)^2")
for a, b in zip(uz, rz):
    print(f"{a:5.2f}    {b:.3f}        {1 - (np.sin(np.pi * a) / (np.pi * a)) ** 2:.3f}")

The density of pairs at distance $u$ (counting each pair once with the later zero to the right, so the target is the right half of Corollary 8.16) starts near $0$, rises like $\frac{\pi^2}3u^2$, and then stays close to $1$ from $u\approx1$ on, as Montgomery's formula says; the wobbles of about $\pm0.1$ are sampling noise from only $500$ zeros. Note the unfolding: with $\log\frac\gamma{2\pi}$ instead of $\log\frac\gamma{2\pi e}$ the mean spacing would be about $1.2$ at this height and the whole curve would sit near $0.8$. This is Montgomery's conjecture, seen with half a thousand zeros.

In [ ]:
from scipy.linalg import toeplitz
n, alpha = 10, 0.35
c = np.array([1 - alpha / np.pi] + [-np.sin(m * alpha) / (np.pi * m) for m in range(1, n)])
det = np.linalg.det(toeplitz(c))
hits = 0; T = 20000
for _ in range(T):
    th = np.angle(np.linalg.eigvals(haar_unitary(n)))
    hits += np.all(np.abs(th) >= alpha)
print(f"Toeplitz determinant: {det:.4f}   simulation: {hits / T:.4f}   independent points: {(1 - alpha / np.pi) ** n:.4f}")

The arc of half-angle $0.35$ contains on average $n\frac\alpha\pi\approx1.11$ eigenvalues. Heine's determinant predicts the probability that it is empty, and the simulation agrees with it to within sampling error. For independent points the probability would be noticeably larger: independent points leave gaps, while eigenvalues fill them. The gap probability of the *sine kernel*, the limit of these determinants, is a Fredholm determinant computed by Jimbo, Miwa, Môri and Sato in 1980 through a Painlevé equation, and it is the distribution that Odlyzko compared with the spacings of zeta zeros.